# ASSIGNMENT 04: NGHIÊN CỨU ĐỐI CHUẨN BASELINE CNN VÀ IMPROVED CNN
**Học phần:** Phát triển các hệ thống thông minh (PGS. TS. Trần Đình Quế)  
**Sinh viên:** Nguyễn Tuấn Tài - **MSSV:** B23DCCN731 - **Lớp:** 06  

Nghiên cứu đối sánh theo Slide 15, 16 và Tutorial mục 45:
- M1 (Baseline CNN - 3 tầng): Conv1 -> ReLU -> Conv2 -> ReLU -> Pool -> Flatten -> Dense
- M2 (Improved CNN - Nâng cấp): M1 + Batch Normalization + Dropout + Residual / Skip Connection


In [ ]:
# Cell [1]: Định nghĩa Baseline CNN vs Improved CNN
import torch
import torch.nn as nn
import json

class BaselineCNN(nn.Module):
    def __init__(self, in_features=8):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv1d(1, 16, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.Conv1d(16, 16, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool1d(2),
            nn.Flatten(),
            nn.Linear(16 * (in_features // 2), 8),
            nn.ReLU(),
            nn.Linear(8, 1)
        )
    def forward(self, x):
        return self.net(x)

class ImprovedCNN(nn.Module):
    def __init__(self, in_features=8):
        super().__init__()
        self.conv1 = nn.Conv1d(1, 16, kernel_size=3, padding=1)
        self.bn1 = nn.BatchNorm1d(16)
        self.relu = nn.ReLU()
        self.conv2 = nn.Conv1d(16, 16, kernel_size=3, padding=1)
        self.bn2 = nn.BatchNorm1d(16)
        self.pool = nn.MaxPool1d(2)
        self.dropout = nn.Dropout(0.2)
        self.head = nn.Sequential(
            nn.Flatten(),
            nn.Linear(16 * (in_features // 2), 16),
            nn.BatchNorm1d(16),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(16, 1)
        )
    def forward(self, x):
        c1 = self.relu(self.bn1(self.conv1(x)))
        c2 = self.relu(self.bn2(self.conv2(c1)) + c1) # Skip connection
        p = self.dropout(self.pool(c2))
        return self.head(p)

print("Baseline CNN Params:", sum(p.numel() for p in BaselineCNN().parameters()))
print("Improved CNN Params:", sum(p.numel() for p in ImprovedCNN().parameters()))


In [ ]:
# Cell [2]: Bảng đối chuẩn kết quả thực nghiệm trên 3 Bộ Dữ Liệu
import pandas as pd
import os
with open('../ass4_experiments/artifacts/diabetes_cnn_metrics.json') as f:
    m_diab = json.load(f)
with open('../ass4_experiments/artifacts/house_cnn_metrics.json') as f:
    m_house = json.load(f)
with open('../ass4_experiments/artifacts/customer_cnn_metrics.json') as f:
    m_cust = json.load(f)

print("=== DIABETES (1D TABULAR) ===")
print(pd.DataFrame(m_diab).T[['Accuracy', 'Precision', 'Recall', 'F1', 'ROC-AUC', 'Params', 'Time']])

print("
=== HOUSE PRICE (1D REGRESSION) ===")
print(pd.DataFrame(m_house).T[['MAE', 'RMSE', 'R2', 'Params', 'Time']])

print("
=== CUSTOMER REVIEWS (1D TEXT) ===")
print(pd.DataFrame(m_cust).T[['Accuracy', 'Macro Precision', 'Macro Recall', 'Macro F1', 'Params', 'Time']])
